In [0]:
storage_account = 'salespipelineadls'
storage_key = 'YOUR_KEY_HERE'

spark.conf.set(
    f'fs.azure.account.key.{storage_account}.dfs.core.windows.net',
    storage_key
)

# Define paths — use these throughout the notebook
bronze_path = f'abfss://bronze@{storage_account}.dfs.core.windows.net/sales_orders/'
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/sales_orders/'

print('Config set. Paths defined.')


Config set. Paths defined.


In [0]:
# Import libraries and read bronze

from pyspark.sql import functions as F
from pyspark.sql.types import *

# Read ALL parquet files in the bronze/sales_orders/ folder
# Spark automatically combines multiple parquet files into one DataFrame
df_bronze = spark.read.parquet(bronze_path)

bronze_count = df_bronze.count()
print(f'Bronze rows read: {bronze_count}')
print(f'Columns: {df_bronze.columns}')

# Preview first 5 rows
display(df_bronze.limit(5))


Bronze rows read: 30000
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'total_amount', 'order_date', 'ship_date', 'status', 'region', 'updated_at']


order_id,customer_id,product_id,quantity,unit_price,discount_pct,total_amount,order_date,ship_date,status,region,updated_at
1,16,5,2,350.00,8.00,644.00,2022-05-07,2022-05-11,Shipped,East,2026-04-09T12:23:10.82Z
2,1,10,1,1800.00,14.00,1548.00,2022-06-05,2022-06-10,Cancelled,West,2026-04-09T12:23:10.823Z
3,5,2,1,1200.00,2.00,1176.00,2023-11-13,2023-11-19,Cancelled,East,2026-04-09T12:23:10.827Z
4,7,1,5,75000.00,12.00,330000.00,2022-10-12,2022-10-18,Delivered,East,2026-04-09T12:23:10.827Z
5,16,6,9,4500.00,2.00,39690.00,2023-02-04,2023-02-08,Shipped,West,2026-04-09T12:23:10.827Z


In [0]:
# STEP 1: Remove duplicate orders
# If pipeline ran twice for same date, same order appears twice
df_dedup = df_bronze.dropDuplicates(['order_id'])
print(f'After dedup: {df_dedup.count()} rows')
print(f'Duplicates removed: {bronze_count - df_dedup.count()}')


# STEP 2: Remove rows where critical columns are null
df_no_nulls = df_dedup.filter(
    F.col('order_id').isNotNull() &
    F.col('customer_id').isNotNull() &
    F.col('product_id').isNotNull() &
    F.col('order_date').isNotNull() &
    F.col('total_amount').isNotNull()
)
print(f'After null removal: {df_no_nulls.count()} rows')



# STEP 3: Remove impossible values
df_valid = df_no_nulls.filter(
    (F.col('total_amount') > 0) &
    (F.col('quantity') > 0) &
    (F.col('unit_price') > 0)
)
print(f'After value validation: {df_valid.count()} rows')


After dedup: 10000 rows
Duplicates removed: 20000
After null removal: 10000 rows
After value validation: 10000 rows


In [0]:
# These new columns make analysis easier in gold layer
df_silver = df_valid \
    .withColumn('order_year',    F.year('order_date')) \
    .withColumn('order_month',   F.month('order_date')) \
    .withColumn('order_quarter', F.quarter('order_date')) \
    .withColumn('order_dayofweek', F.dayofweek('order_date')) \
    .withColumn('revenue',
        F.round(
            F.col('quantity') * F.col('unit_price') *
            (1 - F.col('discount_pct') / 100), 2
        )
    ) \
    .withColumn('is_cancelled',
        F.when(F.col('status') == 'Cancelled', 1).otherwise(0)
    ) \
    .withColumn('days_to_ship',
        F.datediff(F.col('ship_date'), F.col('order_date'))
    ) \
    .withColumn('ingested_at', F.current_timestamp())

print(f'Silver columns: {df_silver.columns}')
display(df_silver.limit(5))

Silver columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'total_amount', 'order_date', 'ship_date', 'status', 'region', 'updated_at', 'order_year', 'order_month', 'order_quarter', 'order_dayofweek', 'revenue', 'is_cancelled', 'days_to_ship', 'ingested_at']


order_id,customer_id,product_id,quantity,unit_price,discount_pct,total_amount,order_date,ship_date,status,region,updated_at,order_year,order_month,order_quarter,order_dayofweek,revenue,is_cancelled,days_to_ship,ingested_at
148,1,6,9,4500.00,17.00,33615.00,2023-10-16,2023-10-23,Cancelled,West,2026-04-09T12:23:10.963Z,2023,10,4,2,33615.00,1,7,2026-04-14T15:26:02.078Z
463,17,6,6,4500.00,19.00,21870.00,2022-08-24,2022-08-28,Cancelled,South,2026-04-09T12:23:11.267Z,2022,8,3,4,21870.00,1,4,2026-04-14T15:26:02.078Z
471,8,9,3,3200.00,7.00,8928.00,2023-02-02,2023-02-08,Shipped,East,2026-04-09T12:23:11.28Z,2023,2,1,5,8928.00,0,6,2026-04-14T15:26:02.078Z
496,9,8,3,2800.00,15.00,7140.00,2022-02-10,2022-02-16,Delivered,East,2026-04-09T12:23:11.3Z,2022,2,1,5,7140.00,0,6,2026-04-14T15:26:02.078Z
833,9,6,5,4500.00,12.00,19800.00,2023-07-26,2023-08-02,Shipped,East,2026-04-09T12:23:11.697Z,2023,7,3,4,19800.00,0,7,2026-04-14T15:26:02.078Z


In [0]:
# Write as Delta format — this is what makes it Delta Lake
# partitionBy creates subfolders by year and month
# This makes queries like WHERE order_year=2022 very fast
df_silver.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .save(silver_path)

print('Silver Delta table written successfully!')

Silver Delta table written successfully!


In [0]:
# Read back the silver table to confirm it worked
df_check = spark.read.format('delta').load(silver_path)

print(f'Silver row count: {df_check.count()}')
print(f'Silver columns: {df_check.columns}')

# Check data by year
display(df_check.groupBy('order_year').count().orderBy('order_year'))

Silver row count: 10000
Silver columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'total_amount', 'order_date', 'ship_date', 'status', 'region', 'updated_at', 'order_year', 'order_month', 'order_quarter', 'order_dayofweek', 'revenue', 'is_cancelled', 'days_to_ship', 'ingested_at']


order_year,count
2022,4973
2023,5027
